# LCO Backfill Demo (backfill_lco_observations)

This notebook demonstrates `solsys_code/management/commands/backfill_lco_observations.py`,
the campaign-agnostic backfill command that creates or updates `ObservationRecord`s (and,
where needed, non-sidereal `Target`s and `ObservationGroup`s) from the LCO Observation
Portal's "Get All RequestGroups" API.

It exists alongside `backfill_lco_observation_records` (see
`backfill_lco_observation_records_demo` / the runbook for that command) but has a different
contract: no campaign is required, an already-existing `ObservationRecord` is updated in
place instead of skipped, an unmatched target is built as a non-sidereal `Target` from its
own orbital elements (never a sidereal field target), and a multi-request `RequestGroup` is
linked into a reusable `ObservationGroup`.

It demonstrates, in order:

- A `--dry-run` pass over a hand-built two-request `RequestGroup` payload, showing the
  summary report with nothing written to the database
- A real pass creating both `ObservationRecord`s, the non-sidereal demo `Target`, and the
  `ObservationGroup` linking them
- A second real pass over an advanced-state, narrowed-schedule payload, printing the
  before/after row for the request whose state and observed block moved, and confirming the
  second request (whose data is unchanged) is left untouched -- both the in-place update and
  the no-churn `unchanged` count are visible
- A cleanup cell deleting every row the notebook created, so it is safely re-runnable

This notebook lives in `pre_executed/` because it is **DB-dependent** (it creates real
`Target`/`ObservationRecord`/`ObservationGroup` rows in the local dev database) and makes no
live network call -- the LCO portal's `GET /api/requestgroups/` response and the
`LCOFacility.get_observation_status()` observed-block lookup are both mocked with a small,
hand-built payload, so a reader can see exactly what the portal's response shape looks like.


## Django setup

Standard boilerplate to make `src.fomo.settings` importable from this notebook's location
(`docs/notebooks/pre_executed/` -- three levels under the repo root, so `parents[2]` gives
the repo root) and to allow synchronous ORM calls inside Jupyter's async event loop.


In [1]:
import os
import sys
from pathlib import Path

import django

# Ensure the repo root is on sys.path so `src.fomo.settings` is importable
# when this notebook is executed from docs/notebooks/pre_executed/.
# NOTE: parents[2] is correct only when the Jupyter kernel CWD is
# docs/notebooks/pre_executed/. Start Jupyter from that directory, or
# adjust the index if you launch from the repo root.
repo_root_path = Path.cwd().resolve().parents[2]
if not (repo_root_path / 'manage.py').exists():
    raise RuntimeError(f'No manage.py at {repo_root_path}; run Jupyter from docs/notebooks/pre_executed/')
repo_root = str(repo_root_path)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'src.fomo.settings')

# Jupyter's ipykernel runs inside an asyncio event loop, but Django's ORM is
# sync-only by default and refuses to run there; this opts back in.
os.environ.setdefault('DJANGO_ALLOW_ASYNC_UNSAFE', 'true')

django.setup()

## Build a hand-built RequestGroup payload

The fixture below is a single `RequestGroup` (id `900100`) carrying **two** requests, so the
same run also demonstrates `ObservationGroup` linking (D-D). Each request's target is a
complete `ORBITAL_ELEMENTS` dict using the LCO/OCS wire-key spelling (`orbinc`,
`longascnode`, `argofperih`, `meandist`, `meananom`, `dailymot`/`eccentricity`, `epochofel`)
that `_extract_orbital_elements()` maps onto TOM `Target` field names (D-E). Neither request
carries an embedded `'observations'` block, so both go through the D-B live fallback
(`LCOFacility.get_observation_status()`), which is mocked below rather than the embedded-block
path -- the more realistic shape for a real portal response, since the RequestGroups listing
endpoint does not itself embed per-request observation blocks.


In [2]:
DEMO_PROPOSAL = 'BACKFILL-DEMO-2026A'
DEMO_TARGET_NAME = 'Backfill Demo Object'
DEMO_GROUP_NAME_FRAGMENT = 'Backfill Demo Group'


def _orbital_elements():
    return {
        'scheme': 'MPC_MINOR_PLANET',
        'orbinc': 5.2,
        'longascnode': 80.1,
        'argofperih': 300.4,
        'meandist': 2.2,
        'meananom': 12.3,
        'eccentricity': 0.15,
        'epochofel': 58600.0,
    }


def _configuration(target_name):
    return {
        'type': 'EXPOSE',
        'instrument_type': '1M0-SCICAM-SINISTRO',
        'instrument_configs': [{'exposure_time': 30.0, 'exposure_count': 1}],
        'target': {'name': target_name, 'type': 'ORBITAL_ELEMENTS', **_orbital_elements()},
    }


def _request(request_id, state):
    return {
        'id': request_id,
        'state': state,
        'windows': [{'start': '2026-07-01T00:00:00', 'end': '2026-07-02T00:00:00'}],
        'configurations': [_configuration(DEMO_TARGET_NAME)],
    }


def _request_group(requests):
    return {
        'id': 900100,
        'name': DEMO_GROUP_NAME_FRAGMENT,
        'proposal': DEMO_PROPOSAL,
        'state': 'PENDING',
        'created': '2026-07-01T00:00:00Z',
        'requests': requests,
    }


first_pass_request_group = _request_group([_request(900101, state='PENDING'), _request(900102, state='COMPLETED')])
print('Requests in demo RequestGroup:', [r['id'] for r in first_pass_request_group['requests']])

Requests in demo RequestGroup: [900101, 900102]


## Mocking helpers

No live network call is made anywhere in this notebook. `make_request` (the paging call
site) is patched to return the fixture payload above; `LCOFacility.get_observation_status`
(the D-B fallback) is patched with a per-observation-id schedule lookup table, so the two
requests can carry independently evolving observed-block times across the two passes below.


In [3]:
import io
from unittest.mock import MagicMock, patch

from django.core.management import call_command

FIRST_PASS_SCHEDULE = {
    '900101': {'scheduled_start': '2026-07-01T00:10:00+00:00', 'scheduled_end': '2026-07-01T00:20:00+00:00'},
    '900102': {'scheduled_start': '2026-07-05T00:10:00+00:00', 'scheduled_end': '2026-07-05T00:20:00+00:00'},
}


def _page_response(results):
    response = MagicMock()
    response.json.return_value = {'count': len(results), 'next': None, 'previous': None, 'results': results}
    return response


def _status_lookup(schedule_table):
    def _get_observation_status(observation_id):
        times = schedule_table[observation_id]
        return {'state': 'na (unused -- status comes from the request payload itself)', **times}

    return _get_observation_status

## `--dry-run` pass: report only, nothing written

With `--dry-run`, the command still resolves every decision (target match/build, schedule)
but performs no `save()`/`create()`/`add()` -- and the D-B live fallback lookup is skipped
entirely, so `get_observation_status` is never called in this cell.

The dry-run counts below equal what the following real pass over the same payload will
report -- `targets would create: 1` because both requests share `DEMO_TARGET_NAME`, a
target FOMO doesn't have yet, and the same-invocation de-dup counts it once, matching how
a real run saves the target on the first request and matches it on the second. The two
schedule-path counters, `embedded blocks` and `fallback lookups needed`, reveal which
schedule path the portal actually exercises for this payload without any extra HTTP call
beyond the initial listing -- neither request here carries an embedded `observations`
block, so both fall back, giving `embedded blocks: 0, fallback lookups needed: 2`. The one
caveat: a request needing the live fallback lookup has its schedule compared by a real run
but not by a dry run (the lookup that would produce a schedule to compare is skipped), so
such a record can be reported `unchanged` by a dry run when only its schedule times would
actually move.


In [4]:
from tom_observations.models import ObservationGroup, ObservationRecord
from tom_targets.models import Target

dry_run_stdout, dry_run_stderr = io.StringIO(), io.StringIO()
with (
    patch(
        'solsys_code.management.commands.backfill_lco_observations.make_request',
        return_value=_page_response([first_pass_request_group]),
    ) as mock_make_request,
    patch('tom_observations.facilities.lco.LCOFacility.get_observation_status') as mock_get_observation_status,
):
    call_command(
        'backfill_lco_observations',
        '--proposal',
        DEMO_PROPOSAL,
        '--dry-run',
        stdout=dry_run_stdout,
        stderr=dry_run_stderr,
    )

print('stdout:', dry_run_stdout.getvalue())
if dry_run_stderr.getvalue():
    print('stderr:', dry_run_stderr.getvalue())

print('get_observation_status called during --dry-run:', mock_get_observation_status.called)
print(
    'ObservationRecords written:',
    ObservationRecord.objects.filter(facility='LCO', observation_id__in=['900101', '900102']).count(),
)
print('Demo Target written:', Target.objects.filter(name=DEMO_TARGET_NAME).exists())
print(
    'Demo ObservationGroup written:',
    ObservationGroup.objects.filter(name__startswith=DEMO_GROUP_NAME_FRAGMENT).exists(),
)

stdout: Would create target 'Backfill Demo Object'; would create ObservationRecord observation_id='900101' status='PENDING'.
Would reuse target 'Backfill Demo Object'; would create ObservationRecord observation_id='900102' status='COMPLETED'.
Would create ObservationGroup 'Backfill Demo Group (900100)'.
requestgroups seen: 1, would create: 2, would update: 0, unchanged: 0, skipped: 0, targets would create: 1, groups would create: 1, groups would reuse: 0, embedded blocks: 0, fallback lookups needed: 2, block lookups failed: n/a (dry-run)

get_observation_status called during --dry-run: False
ObservationRecords written: 0
Demo Target written: False
Demo ObservationGroup written: False


## Real pass: create the records, the non-sidereal target, and the group

The same payload, without `--dry-run`. `DEMO_TARGET_NAME` isn't a Target FOMO already knows
about, so it gets built as a non-sidereal `Target` from the orbital elements in the fixture
above -- never a sidereal one (D-G). Because the `RequestGroup` carries two requests, the
command also creates one `ObservationGroup` linking both resulting records (D-D).


In [5]:
real_stdout, real_stderr = io.StringIO(), io.StringIO()
with (
    patch(
        'solsys_code.management.commands.backfill_lco_observations.make_request',
        return_value=_page_response([first_pass_request_group]),
    ),
    patch(
        'tom_observations.facilities.lco.LCOFacility.get_observation_status',
        side_effect=_status_lookup(FIRST_PASS_SCHEDULE),
    ),
):
    call_command('backfill_lco_observations', '--proposal', DEMO_PROPOSAL, stdout=real_stdout, stderr=real_stderr)

print('stdout:', real_stdout.getvalue())
if real_stderr.getvalue():
    print('stderr:', real_stderr.getvalue())

Target post save hook: Backfill Demo Object created: True


Observation change state hook: Backfill Demo Object @ LCO from None to PENDING


Observation change state hook: Backfill Demo Object @ LCO from None to COMPLETED


stdout: requestgroups seen: 1, created: 2, updated: 0, unchanged: 0, skipped: 0, targets created: 1, groups created: 1, groups reused: 0, embedded blocks: 0, fallback lookups needed: 2, block lookups failed: 0



## Inspect what the real pass created

In [6]:
demo_target = Target.objects.get(name=DEMO_TARGET_NAME)
print('target type       :', demo_target.type)
print('target scheme      :', demo_target.scheme)
print('target inclination :', demo_target.inclination)

record_101 = ObservationRecord.objects.get(facility='LCO', observation_id='900101')
record_102 = ObservationRecord.objects.get(facility='LCO', observation_id='900102')
print('900101 status/start/end:', record_101.status, record_101.scheduled_start, record_101.scheduled_end)
print('900102 status/start/end:', record_102.status, record_102.scheduled_start, record_102.scheduled_end)

demo_group = ObservationGroup.objects.get(name__startswith=DEMO_GROUP_NAME_FRAGMENT)
print('group name         :', demo_group.name)
print('group record ids   :', sorted(r.observation_id for r in demo_group.observation_records.all()))

if demo_target.type == Target.NON_SIDEREAL and not Target.objects.filter(type=Target.SIDEREAL).exists():
    print('PASS: demo target is non-sidereal, and no sidereal target exists anywhere')

target type       : NON_SIDEREAL
target scheme      : MPC_MINOR_PLANET
target inclination : 5.2
900101 status/start/end: PENDING 2026-07-01 00:10:00+00:00 2026-07-01 00:20:00+00:00
900102 status/start/end: COMPLETED 2026-07-05 00:10:00+00:00 2026-07-05 00:20:00+00:00
group name         : Backfill Demo Group (900100)
group record ids   : ['900101', '900102']


## Second real pass: an advanced state and a narrowed observed block

Request `900101` advances from `PENDING` to `COMPLETED`, and its mocked observed block
narrows (`00:10-00:20` -> `00:15-00:18`) -- both are picked up by the second pass, updating
the existing `ObservationRecord` in place rather than creating a second one. Request
`900102`'s state and schedule are unchanged, so its row is left untouched (`modified`
timestamp unchanged) and the summary line counts it under `unchanged`, not `updated`.


In [7]:
before_status = record_101.status
before_start = record_101.scheduled_start
before_end = record_101.scheduled_end
before_modified_102 = record_102.modified

second_pass_request_group = _request_group([_request(900101, state='COMPLETED'), _request(900102, state='COMPLETED')])

SECOND_PASS_SCHEDULE = {
    '900101': {'scheduled_start': '2026-07-01T00:15:00+00:00', 'scheduled_end': '2026-07-01T00:18:00+00:00'},
    '900102': FIRST_PASS_SCHEDULE['900102'],  # unchanged -- proves the no-churn path
}

second_stdout, second_stderr = io.StringIO(), io.StringIO()
with (
    patch(
        'solsys_code.management.commands.backfill_lco_observations.make_request',
        return_value=_page_response([second_pass_request_group]),
    ),
    patch(
        'tom_observations.facilities.lco.LCOFacility.get_observation_status',
        side_effect=_status_lookup(SECOND_PASS_SCHEDULE),
    ),
):
    call_command('backfill_lco_observations', '--proposal', DEMO_PROPOSAL, stdout=second_stdout, stderr=second_stderr)

print('stdout:', second_stdout.getvalue())

record_101.refresh_from_db()
record_102.refresh_from_db()

print('900101 before -> after')
print('  status:', before_status, '->', record_101.status)
print('  start :', before_start, '->', record_101.scheduled_start)
print('  end   :', before_end, '->', record_101.scheduled_end)
print()
print('900102 modified timestamp unchanged:', record_102.modified == before_modified_102)

if record_101.status == 'COMPLETED' and record_101.scheduled_start.isoformat() == '2026-07-01T00:15:00+00:00':
    print('PASS: 900101 updated in place (no second row created)')
if record_102.modified == before_modified_102:
    print('PASS: 900102 left unchanged (no-churn re-run)')
print(
    'records still exactly 2:',
    ObservationRecord.objects.filter(facility='LCO', observation_id__in=['900101', '900102']).count() == 2,
)
print(
    'groups still exactly 1:', ObservationGroup.objects.filter(name__startswith=DEMO_GROUP_NAME_FRAGMENT).count() == 1
)

Observation change state hook: Backfill Demo Object @ LCO from PENDING to COMPLETED


stdout: requestgroups seen: 1, created: 0, updated: 1, unchanged: 1, skipped: 0, targets created: 0, groups created: 0, groups reused: 1, embedded blocks: 0, fallback lookups needed: 2, block lookups failed: 0

900101 before -> after
  status: PENDING -> COMPLETED
  start : 2026-07-01 00:10:00+00:00 -> 2026-07-01 00:15:00+00:00
  end   : 2026-07-01 00:20:00+00:00 -> 2026-07-01 00:18:00+00:00

900102 modified timestamp unchanged: True
PASS: 900101 updated in place (no second row created)
PASS: 900102 left unchanged (no-churn re-run)
records still exactly 2: True
groups still exactly 1: True


## Cleanup

This notebook is DB-dependent and creates real rows in the local dev database. Clean up
everything created above so re-running the notebook starts from a clean slate.


In [8]:
ObservationGroup.objects.filter(name__startswith=DEMO_GROUP_NAME_FRAGMENT).delete()
ObservationRecord.objects.filter(facility='LCO', observation_id__in=['900101', '900102']).delete()
Target.objects.filter(name=DEMO_TARGET_NAME).delete()

print(
    'records remaining:',
    ObservationRecord.objects.filter(facility='LCO', observation_id__in=['900101', '900102']).count(),
)
print('groups remaining :', ObservationGroup.objects.filter(name__startswith=DEMO_GROUP_NAME_FRAGMENT).count())
print('target remaining :', Target.objects.filter(name=DEMO_TARGET_NAME).exists())

Total removed orphan object permissions instances: 0


records remaining: 0
groups remaining : 0
target remaining : False
